# Dataset

The dataset used is a compilation of BBC news articles ranging in different topics (politics, business, entertainment, sport and tech) from here: https://huggingface.co/datasets/gopalkalpande/bbc-news-summary

In [ ]:
# Function to load and preprocess the BBC News Summary dataset
def preprocessing_corpus(sourcepath, word_or_character="word", N=2):
	import pandas as pd

	df = pd.read_csv(sourcepath)
	df.columns = ["Article", "Category"]

	# Convert all text to lowercase
	#df['Article'] = df['Article'].str.lower()

	# Replace unicode literal for pound sign
	df = df.replace("xc2xa3", "£", regex=True)

	if word_or_character == "word":
		# Add String "<s> " at the beginning of each text
		df['Article'] = "<s> " + df['Article']

		# For all texts start cleaning by replacing single, double or triple full stops characters with "</s> <s>"
		df = df.replace(r"\.{1,3}", " </s> <s> ", regex=True)
		df = df.replace(r"\!{1,3}", " </s> <s> ", regex=True)
		df = df.replace(r"\?{1,3}", " </s> <s> ", regex=True)
		df = df.replace(r"\ {2,3}", " </s> <s> ", regex=True)

		# Remove all appearances of empty sentences "<s> </s>" and remove the last sentence start token "<s>" at the end of each text
		df = df.replace("<s> </s>", "", regex=True)
		df['Article'] = df['Article'].str[:-6]

		# Remove all appearances of the characters ( ) [ ] { } , " : ; from the texts
		chars_to_remove = [r"\(", r"\)", r"\[", r"\]", r"\{", r"\}", r"\,", r'"', r"\:", r"\;"]
		for char in chars_to_remove:
			df = df.replace(char, "", regex=True)

		# Replace multiple spaces (1, 2 or 3) with a single space	
		df = df.replace(r"\ {1,3}", " ", regex=True)

		# Split each text into a list of words
		words = lambda text: text.split(" ")
		df['Article'] = df['Article'].apply(words)

	else:
		# Replace multiple spaces (1, 2 or 3) with a single space	
		df = df.replace(r"\ {2,3}", " ", regex=True)
		
		# Split each text into a list of characters
		chars = lambda text: list(text)
		df['Article'] = df['Article'].apply(chars)

	# Generate n-grams and store them in a new DataFrame		
	ngram_list = []
	category_list = []

	for index, row in df.iterrows():
		category = row['Category']
		article = row['Article']
		
		# Generate all n-grams for this article at once
		for i in range(len(article) - N + 1):
			ngram = article[i:i+N]
			ngram_list.append(ngram)
			category_list.append(category)

		print(f"Processing corpus for {N}-gram: {index/(len(df)-1)*100:.2f}% processed", end="\r")
	
	ngrams = pd.DataFrame({'N-gram': ngram_list, 'Category': category_list})
    
	return ngrams

In [ ]:
# Preprocess text corpus and store ngrams to a new CSV file
N = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
modeltype = {"word", "character"}
sourcepath = "./../Dataset/bbc_data.csv"
targetfolder = "./../Dataset_NGrams/"
for n in N:
    for mt in modeltype:
        ngrams = preprocessing_corpus(sourcepath, mt, n)
        ngrams.to_csv(f"{targetfolder}bbc_{mt}_{n}grams.csv", index=False)